# SFD 跨域半监督联邦学习算法分析 (SFD Semi-Supervised FL Analysis)

本 Notebook 用于对 **FedMatch**, **FedPLN-SSL**, **FedAvg-SL (Oracle 全监督上限)**, **FedAvg-L (纯源域有标签下限)**, **FixMatch-LPL** 和 **FixMatch-GPL** 等算法在跨域半监督（SFD）场景下的性能进行全面对比与单算法深入剖析。

### 核心评估维度：
1. **全局准确率 (Overall Test Accuracy)**
2. **有标签源域准确率 (Source Labeled Accuracy)**
3. **无标签目标域准确率 (Target Unlabeled Accuracy)**
4. **原型准确率 (Prototype Accuracy / 跨域原型分类)**
> **读取语义说明**：结果加载为严格报错模式——结果文件缺失/损坏、命名参数不一致都会直接抛出异常终止，不做静默跳过；未跑完的算法请先注释掉 `experiments` 中对应条目。


In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# 添加项目根目录到 Python 路径
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import ResultLoader, plot_sfd_results

## 1. 实验配置与算法定义
配置 SFD 场景下的公共参数（如源域 `clean`、目标域 `rotate_cutout`）及各算法特定超参数。

In [ ]:
# --- SFD 公共配置 ---
# 注意：momentum/weight_decay/unlabeled_ratio/lam/confidence 会编码进结果文件名，
# 必须与训练时取值完全一致（当前与 configs/default.yaml 默认值对齐）。
common_args = {
    "dataset": "cifar10_dg",
    "model": "cnn",
    "partition": "dirichlet",
    "alpha": 0.1,
    "num_clients": 20,
    "ssl": "sfd",
    "label_domain": "clean",
    "unlabel_domain": "rotate_cutout",
    "label_ratio": 0.5,
    "unlabeled_ratio": 2,
    "lam": 1.0,
    "confidence": 0.95,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs": 1,
    "batch_size": 64,
    "lr": 0.01,
}

# --- 算法参数定义 ---
# FedMatch / FedTest 的超参与 configs/algorithms.yaml 对齐（值被编码进文件名，必须一致）。
experiments = {
    "FedAvg-SL (Oracle Upper Bound)": ("fedavg_sl", {}),
    "FedAvg-L (Lower Bound)": ("fedavg_l", {}),
    "FedAvg-LPL": ("fedavg_lpl", {}),
    "FedAvg-GPL": ("fedavg_gpl", {}),
    "FedMatch": (
        "fedmatch",
        {
            "confidence": 0.5,
            "h_interval": 10,
            "num_helpers": 2,
            "lambda_s": 10.0,
            "lambda_i": 0.01,
            "lambda_a": 0.01,
            "lambda_l2": 10.0,
            "lambda_l1": 1e-05,
            "l1_thres": 5e-05,
            "delta_thres": 5e-05,
            "psi_factor": 0.2,
        },
    ),
    "FedTest": (
        "fedtest",
        {
            "lambda_u": 0.2,
            "temperature": 0.2,
            "source_proto_weight": 0.5,
            "prior_strength": 0.5,
            "reliability_power": 2.0,
            "reliability_floor": 0.05,
            "diagnostic_confidence": 0.8,
            "source_proto_momentum": 0.5,
            "proto_momentum": 0.9,
            "proto_anchor": 0.1,
            "assignment_prior_momentum": 0.9,
            "teacher_momentum": 0.99,
            "warmup_rounds": 10,
        },
    ),
}

loader = ResultLoader(base_dir="../results")


## 2. 多算法综合对比 (Multi-Algorithm Comparison)
横向对比各算法在 **Overall / Source Labeled / Target Unlabeled** 上的准确率曲线及原型性能。

In [ ]:
# 批量加载实验结果（文件缺失/命名不一致直接抛错，不做静默跳过）
loaded_results = {}
for label, (algo_name, kwargs) in experiments.items():
    merged_args = {**common_args, **kwargs}
    loaded_results[label] = loader.load(algo_name, specific_run=0, **merged_args)

# 绘制三子图对比 (Overall | Source | Target) 并展示原型曲线
plot_sfd_results(
    loaded_results,
    x_lim=200,
    title=f"SFD Comparison on {common_args['dataset']} ({common_args['label_domain']} -> {common_args['unlabel_domain']})",
    show_proto=True,
)
